# Model Input Testing

Goal: fill a transaction's fields by hand, run the cell, and see if the deployed XGBoost model predicts **fraud** or **not fraud**. Use this to sanity-check the model before wiring it into FastAPI.

**Caveat:** 8 of the 36 features are user-history derived (`tx_count_24h`, `user_mean_amt`, `time_since_last_tx`, etc.). For a hand-typed single transaction we fill these with neutral defaults — predictions are directionally correct but not as sharp as a real production lookup. Good enough for input-flow sanity testing.

## 1. Load model + encoders + metadata

In [ ]:
import json
import pickle
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings('ignore')

OUT_DIR = Path('..') / 'outputs'

with open(OUT_DIR / 'model.pkl', 'rb') as f:
    model = pickle.load(f)

with open(OUT_DIR / 'encoders.pkl', 'rb') as f:
    enc = pickle.load(f)

with open(OUT_DIR / 'model_metadata.json', 'r') as f:
    meta = json.load(f)

FEATURE_ORDER = meta['feature_order']
THRESHOLD     = meta['deploy_threshold']

print(f'Model       : {type(model).__name__}  (deploy = {meta["deploy_model"]})')
print(f'Threshold   : {THRESHOLD}')
print(f'# features  : {len(FEATURE_ORDER)}')
print(f'Encoders    : {list(enc.keys())}')
print(f'Split type  : {meta.get("split", {}).get("type")}   <- should be stratified_random_70_15_15 for v2')

## 2. Valid categorical values

In [ ]:
print('card_brand     :', list(enc['card_brand_encoder'].classes_))
print('use_chip       :', list(enc['chip_map'].keys()))
print('merchant_state : 200 options — common: CA, NY, TX, FL, IL, WA, ...')
print('                 (full list:', list(enc['merchant_state_encoder'].classes_[:15]), '...)')
print('gender         : Male | Female')
print('card_type      : Credit | Debit | Debit (Prepaid)')
print('mcc            : integer MCC code, e.g. 5411 (groceries), 5812 (restaurants),')
print('                 5942 (books), 7995 (gambling), 4829 (wire transfer)')

## 3. The `predict(tx)` function

In [ ]:
def encode_tx(tx: dict) -> pd.DataFrame:
    """Apply the same encoding NB2 v2 uses, then arrange in feature_order."""
    d = dict(tx)

    dt = pd.to_datetime(d['date'])
    d['tx_hour']       = dt.hour
    d['tx_day']        = dt.day
    d['tx_month']      = dt.month
    d['tx_dayofweek']  = dt.dayofweek
    d['tx_is_weekend'] = int(dt.dayofweek >= 5)
    d['tx_is_night']   = int(dt.hour >= 22 or dt.hour <= 5)

    amt = float(d['amount'])
    d['amount_abs']             = abs(amt)
    d['is_negative']            = int(amt < 0)
    d['amount_log']             = float(np.log1p(abs(amt)))
    d['amount_to_limit_ratio']  = min(abs(amt) / (abs(d['credit_limit']) + 1), 10.0)
    d['amount_to_income_ratio'] = min(abs(amt) / (abs(d['yearly_income']) + 1), 5.0)

    d['age_at_tx']           = dt.year - int(d['birth_year'])
    d['years_to_retirement'] = int(d['retirement_age']) - int(d['current_age'])
    d['debt_to_income']      = d['total_debt'] / (abs(d['yearly_income']) + 1)
    d['has_chip_flag']       = int(str(d.get('has_chip', 'no')).lower() == 'yes')

    d['use_chip_enc']  = enc['chip_map'].get(d['use_chip'], -1)
    d['gender_enc']    = int(str(d['gender']).lower() == 'male')
    d['card_type_enc'] = int(str(d['card_type']).lower() == 'credit')
    d['mcc_enc']       = int(d['mcc'])

    brand_classes = list(enc['card_brand_encoder'].classes_)
    d['card_brand_enc'] = (brand_classes.index(d['card_brand'])
                          if d['card_brand'] in brand_classes else 0)

    state_classes = list(enc['merchant_state_encoder'].classes_)
    d['merchant_state_enc'] = (state_classes.index(d['merchant_state'])
                              if d['merchant_state'] in state_classes else 0)

    d.setdefault('time_since_last_tx',  3600.0)
    d.setdefault('tx_count_24h',        2)
    d.setdefault('amount_sum_24h',      d['amount_abs'])
    d.setdefault('tx_count_7d',         10)
    d.setdefault('amount_sum_7d',       d['amount_abs'] * 5)
    d.setdefault('user_mean_amt',       d['amount_abs'])
    d.setdefault('user_std_amt',        d['amount_abs'] * 0.3)
    d.setdefault('amount_vs_user_mean', 1.0)

    row = {col: d.get(col, 0) for col in FEATURE_ORDER}
    return pd.DataFrame([row], columns=FEATURE_ORDER)


def predict(tx: dict, verbose: bool = True) -> dict:
    X = encode_tx(tx)
    proba = float(model.predict_proba(X)[0, 1])
    is_fraud = bool(proba >= THRESHOLD)
    result = {
        'fraud_probability': round(proba, 6),
        'is_fraud':          is_fraud,
        'threshold_used':    THRESHOLD,
        'verdict':           'FRAUD' if is_fraud else 'NOT FRAUD',
    }
    if verbose:
        bar = '#' * int(proba * 40)
        print(f'  prob = {proba:.6f}  {bar}')
        print(f'  thr  = {THRESHOLD}')
        print(f'  >>> {result["verdict"]}')
    return result

## 3b. DIAGNOSTIC — run this if predictions look wrong

If the fraud-looking sample is returning prob ≈ 0 (NOT FRAUD), something is misaligned. This cell prints what's happening end-to-end so we can pinpoint the bug.

In [ ]:
# Build a fraud-loaded tx for diagnostic
tx_diag = {
    'date':            '2019-08-24 03:17:00',
    'amount':          2500.00,
    'use_chip':        'Online Transaction',
    'merchant_state':  'Russia',
    'mcc':             7995,
    'card_brand':      'Visa',
    'card_type':       'Credit',
    'has_chip':        'yes',
    'gender':          'Female',
    'birth_year':      1985,
    'current_age':     34,
    'retirement_age':  65,
    'credit_score':    640,
    'num_credit_cards':2,
    'num_cards_issued':1,
    'credit_limit':    5000,
    'yearly_income':   45000,
    'total_debt':      8000,
    'per_capita_income': 28000,
    'tx_count_24h':       10,
    'amount_sum_24h':     6000.0,
    'user_mean_amt':      40.0,
    'amount_vs_user_mean': 62.5,
    'time_since_last_tx': 180.0,
}

print('=== 1. Metadata sanity check ===')
print('Split type :', meta.get('split', {}).get('type'))
print('Deploy     :', meta['deploy_model'])
print('Threshold  :', THRESHOLD)
xgb_metrics = [m for m in meta['metrics_default'] if m['Model']=='XGBoost']
if xgb_metrics:
    print('Test F1    :', xgb_metrics[0]['F1'], ' (should be ~0.84)')
    print('Test PR-AUC:', xgb_metrics[0]['PR_AUC'], ' (should be ~0.89)')

print('\n=== 2. Encoded feature row for fraud-loaded tx ===')
X = encode_tx(tx_diag)
print(X.T.to_string())

print('\n=== 3. Model feature names vs our column order ===')
model_feats = None
if hasattr(model, 'feature_names_in_'):
    model_feats = list(model.feature_names_in_)
elif hasattr(model, 'get_booster'):
    try:
        model_feats = model.get_booster().feature_names
    except Exception:
        pass

if model_feats:
    print('Model expects first 5 :', model_feats[:5])
    print('Model expects last 3  :', model_feats[-3:])
    print('We pass first 5       :', list(X.columns)[:5])
    print('We pass last 3        :', list(X.columns)[-3:])
    print('Names match exactly?  :', model_feats == list(X.columns))
    missing = [c for c in model_feats if c not in X.columns]
    extra   = [c for c in X.columns if c not in model_feats]
    if missing: print('MISSING in our input  :', missing)
    if extra:   print('EXTRA   in our input  :', extra)
else:
    print('(could not read model feature names — older XGBoost?)')

print('\n=== 4. Raw model probabilities ===')
raw_proba = model.predict_proba(X)
print('raw predict_proba         :', raw_proba)
print('class 0 (not fraud) prob  :', float(raw_proba[0,0]))
print('class 1 (fraud)     prob  :', float(raw_proba[0,1]))
print('\n=== Diagnosis ===')
print('If Test F1 != 0.84 -> wrong model.pkl loaded')
print('If Names match? = False -> column order is off')
print('If row has any unexpected 0 or NaN -> a field was not encoded')
print('If class 1 prob > 0.5 here but predict() said NOT FRAUD -> predict() wrapper bug')

## 4. Sample 1 — typical legitimate transaction

Small grocery purchase, daytime, weekday, home state. Expect a **low** fraud probability.

In [ ]:
tx_legit = {
    'date':            '2019-03-12 14:30:00',
    'amount':          42.50,
    'use_chip':        'Chip Transaction',
    'merchant_state':  'CA',
    'mcc':             5411,
    'card_brand':      'Visa',
    'card_type':       'Debit',
    'has_chip':        'yes',
    'gender':          'Female',
    'birth_year':      1985,
    'current_age':     34,
    'retirement_age':  65,
    'credit_score':    740,
    'num_credit_cards':3,
    'num_cards_issued':1,
    'credit_limit':    8000,
    'yearly_income':   62000,
    'total_debt':      4200,
    'per_capita_income': 38000,
}
print('--- Sample 1: legitimate-looking ---')
predict(tx_legit);

## 5. Sample 2 — REAL fraud pattern (refund abuse)

**Key finding from diagnostic:** Real fraud in the IBM synthetic dataset is **refund abuse** — online transactions with merchant_state='Unknown' and a **negative amount** (the fraudster issues fake refunds to a stolen card). The top features by importance are `use_chip_enc` (40%), `merchant_state_enc` (23%), `mcc_enc` (7%), and `is_negative` (4%).

Large international "gambling-style" transactions do **not** trigger this model — that pattern isn't in the training data. Worth discussing in your thesis Limitations section.

In [ ]:
tx_fraud = {
    'date':            '2019-08-24 19:00:00',
    'amount':          -339.00,                # NEGATIVE = refund-abuse signal
    'use_chip':        'Online Transaction',
    'merchant_state':  'Unknown',              # most real fraud has no merchant state
    'mcc':             3640,                   # one of the real-fraud MCCs
    'card_brand':      'Visa',
    'card_type':       'Credit',
    'has_chip':        'yes',
    'gender':          'Female',
    'birth_year':      1985,
    'current_age':     34,
    'retirement_age':  65,
    'credit_score':    740,
    'num_credit_cards':3,
    'num_cards_issued':1,
    'credit_limit':    8000,
    'yearly_income':   62000,
    'total_debt':      4200,
    'per_capita_income': 38000,
}
print('--- Sample 2: real fraud pattern (refund abuse) ---')
predict(tx_fraud);

## 6. Your turn — fill and run

In [ ]:
tx_test = {
    'date':            '2019-06-15 12:00:00',
    'amount':          100.00,
    'use_chip':        'Chip Transaction',
    'merchant_state':  'NY',
    'mcc':             5812,
    'card_brand':      'Mastercard',
    'card_type':       'Credit',
    'has_chip':        'yes',
    'gender':          'Male',
    'birth_year':      1990,
    'current_age':     29,
    'retirement_age':  65,
    'credit_score':    700,
    'num_credit_cards':2,
    'num_cards_issued':1,
    'credit_limit':    6000,
    'yearly_income':   55000,
    'total_debt':      3000,
    'per_capita_income': 35000,
}
predict(tx_test);

## 7. Tweak experiments — see what drives predictions

In [ ]:
base = dict(tx_legit)   # start from a legit transaction

# Step from legit -> fraud one signal at a time, along the REAL fraud axes
experiments = {
    'baseline (legit grocery)':          base,
    'chip -> online':                    {**base, 'use_chip': 'Online Transaction'},
    '  + state CA -> Unknown':           {**base, 'use_chip': 'Online Transaction',
                                          'merchant_state': 'Unknown'},
    '  + mcc grocery -> 3640':           {**base, 'use_chip': 'Online Transaction',
                                          'merchant_state': 'Unknown', 'mcc': 3640},
    '  + amount +42 -> -339 (refund)':   {**base, 'use_chip': 'Online Transaction',
                                          'merchant_state': 'Unknown', 'mcc': 3640,
                                          'amount': -339.00},
}

print(f'{"experiment":35s}  prob       verdict')
print('-' * 65)
for name, tx in experiments.items():
    r = predict(tx, verbose=False)
    print(f'{name:35s}  {r["fraud_probability"]:.6f}  {r["verdict"]}')

## 8. Interactive demo — type values, get instant verdict

Run this cell and answer the prompts one by one. Press Enter to accept the default shown in [brackets]. Good for a live demo before building FastAPI.

In [ ]:
def ask(label, default):
    """Prompt the user; Enter keeps the default."""
    ans = input(f'{label} [{default}]: ').strip()
    return ans if ans else str(default)

print('=== Fill a transaction (press Enter to keep the [default]) ===\n')

tx = {
    'date':            ask('Date (YYYY-MM-DD HH:MM:SS)', '2019-06-15 12:00:00'),
    'amount':          float(ask('Amount (negative = refund)', '100')),
    'use_chip':        ask('Use chip (Chip Transaction / Swipe Transaction / Online Transaction)',
                           'Online Transaction'),
    'merchant_state':  ask('Merchant state (CA, NY, ... or Unknown)', 'Unknown'),
    'mcc':             int(ask('MCC code (5411 grocery / 5812 restaurant / 3640 / 5311)', '3640')),
    'card_brand':      ask('Card brand (Visa / Mastercard / Amex / Discover)', 'Visa'),
    'card_type':       ask('Card type (Credit / Debit)', 'Credit'),
    'has_chip':        ask('Card has chip (yes/no)', 'yes'),
    'gender':          ask('Gender (Male/Female)', 'Male'),
    'birth_year':      int(ask('Birth year', '1990')),
    'current_age':     int(ask('Current age', '29')),
    'retirement_age':  65,
    'credit_score':    int(ask('Credit score', '700')),
    'num_credit_cards':2,
    'num_cards_issued':1,
    'credit_limit':    float(ask('Credit limit', '6000')),
    'yearly_income':   float(ask('Yearly income', '55000')),
    'total_debt':      3000,
    'per_capita_income': 35000,
}

print('\n=== RESULT ===')
r = predict(tx)
print('\nFull output:', r)

In [ ]:
# Reusable "normal" cardholder profile so we only vary the transaction fields
PROFILE = {
    'card_brand': 'Visa', 'card_type': 'Credit', 'has_chip': 'yes',
    'gender': 'Male', 'birth_year': 1990, 'current_age': 29, 'retirement_age': 65,
    'credit_score': 700, 'num_credit_cards': 2, 'num_cards_issued': 1,
    'credit_limit': 6000, 'yearly_income': 55000, 'total_debt': 3000,
    'per_capita_income': 35000,
}

def case(expected, note, **tx_fields):
    return {'expected': expected, 'note': note,
            'tx': {**PROFILE, **tx_fields}}

# expected = what the verdict SHOULD be.
# NOTE: not every MCC triggers fraud. The model learned exact MCC+context
# combinations — e.g. 3640 and 5300 fire, but 5311 does not on its own.
# "Appears in the fraud sample" is NOT the same as "is a fraud trigger".
CASES = [
    # ----- should be NOT FRAUD -----
    case('NOT FRAUD', 'in-store grocery',      date='2019-03-12 14:30:00', amount=42.50,
         use_chip='Chip Transaction',  merchant_state='CA', mcc=5411),
    case('NOT FRAUD', 'swipe gas station',     date='2019-05-01 08:00:00', amount=60.00,
         use_chip='Swipe Transaction', merchant_state='TX', mcc=5541),
    case('NOT FRAUD', 'online but real state', date='2019-06-15 12:00:00', amount=120.00,
         use_chip='Online Transaction', merchant_state='NY', mcc=5311),
    case('NOT FRAUD', 'chip + unknown state',  date='2019-06-15 12:00:00', amount=120.00,
         use_chip='Chip Transaction',  merchant_state='Unknown', mcc=3640),
    case('NOT FRAUD', 'online + unknown + grocery mcc', date='2019-06-15 12:00:00', amount=80.00,
         use_chip='Online Transaction', merchant_state='Unknown', mcc=5411),
    case('NOT FRAUD', 'online + unknown + mcc5311 (NOT a trigger)', date='2019-07-02 21:00:00', amount=15.00,
         use_chip='Online Transaction', merchant_state='Unknown', mcc=5311),
    case('NOT FRAUD', 'huge amount but normal channel', date='2019-06-15 12:00:00', amount=999999.0,
         use_chip='Chip Transaction',  merchant_state='CA', mcc=5411),

    # ----- should be FRAUD -----
    case('FRAUD', 'online+unknown+mcc3640',          date='2019-06-15 12:00:00', amount=100.00,
         use_chip='Online Transaction', merchant_state='Unknown', mcc=3640),
    case('FRAUD', 'refund abuse (negative amount)',  date='2019-08-24 19:00:00', amount=-339.00,
         use_chip='Online Transaction', merchant_state='Unknown', mcc=3640),
    case('FRAUD', 'online+unknown+mcc5300 small',    date='2019-09-10 02:00:00', amount=7.19,
         use_chip='Online Transaction', merchant_state='Unknown', mcc=5300),
]

print(f'{"#":>2}  {"expected":10s}  {"got":10s}  {"prob":>9s}  result  note')
print('-' * 84)
passed = 0
for i, c in enumerate(CASES, 1):
    r = predict(c['tx'], verbose=False)
    got = r['verdict']
    ok = (got == c['expected'])
    passed += ok
    flag = 'PASS' if ok else '*** FAIL ***'
    print(f'{i:>2}  {c["expected"]:10s}  {got:10s}  {r["fraud_probability"]:.6f}  {flag:5s}  {c["note"]}')

print('-' * 84)
print(f'{passed}/{len(CASES)} cases passed')
if passed == len(CASES):
    print('ALL PASS - model behaves as expected, safe to deploy.')
else:
    print('Some FAIL - inspect those rows before deploying.')

In [ ]:
# Reusable "normal" cardholder profile so we only vary the transaction fields
PROFILE = {
    'card_brand': 'Visa', 'card_type': 'Credit', 'has_chip': 'yes',
    'gender': 'Male', 'birth_year': 1990, 'current_age': 29, 'retirement_age': 65,
    'credit_score': 700, 'num_credit_cards': 2, 'num_cards_issued': 1,
    'credit_limit': 6000, 'yearly_income': 55000, 'total_debt': 3000,
    'per_capita_income': 35000,
}

def case(expected, note, **tx_fields):
    return {'expected': expected, 'note': note,
            'tx': {**PROFILE, **tx_fields}}

# expected = what the verdict SHOULD be
CASES = [
    # ----- should be NOT FRAUD -----
    case('NOT FRAUD', 'in-store grocery',      date='2019-03-12 14:30:00', amount=42.50,
         use_chip='Chip Transaction',  merchant_state='CA', mcc=5411),
    case('NOT FRAUD', 'swipe gas station',     date='2019-05-01 08:00:00', amount=60.00,
         use_chip='Swipe Transaction', merchant_state='TX', mcc=5541),
    case('NOT FRAUD', 'online but real state', date='2019-06-15 12:00:00', amount=120.00,
         use_chip='Online Transaction', merchant_state='NY', mcc=5311),
    case('NOT FRAUD', 'chip + unknown state',  date='2019-06-15 12:00:00', amount=120.00,
         use_chip='Chip Transaction',  merchant_state='Unknown', mcc=3640),
    case('NOT FRAUD', 'online + unknown + grocery mcc', date='2019-06-15 12:00:00', amount=80.00,
         use_chip='Online Transaction', merchant_state='Unknown', mcc=5411),
    case('NOT FRAUD', 'huge amount but normal channel', date='2019-06-15 12:00:00', amount=999999.0,
         use_chip='Chip Transaction',  merchant_state='CA', mcc=5411),

    # ----- should be FRAUD -----
    case('FRAUD', 'online+unknown+mcc3640',          date='2019-06-15 12:00:00', amount=100.00,
         use_chip='Online Transaction', merchant_state='Unknown', mcc=3640),
    case('FRAUD', 'online+unknown+mcc5311',          date='2019-07-02 21:00:00', amount=15.00,
         use_chip='Online Transaction', merchant_state='Unknown', mcc=5311),
    case('FRAUD', 'refund abuse (negative amount)',  date='2019-08-24 19:00:00', amount=-339.00,
         use_chip='Online Transaction', merchant_state='Unknown', mcc=3640),
    case('FRAUD', 'online+unknown+mcc5300 small',    date='2019-09-10 02:00:00', amount=7.19,
         use_chip='Online Transaction', merchant_state='Unknown', mcc=5300),
]

print(f'{"#":>2}  {"expected":10s}  {"got":10s}  {"prob":>9s}  result  note')
print('-' * 80)
passed = 0
for i, c in enumerate(CASES, 1):
    r = predict(c['tx'], verbose=False)
    got = r['verdict']
    ok = (got == c['expected'])
    passed += ok
    flag = 'PASS' if ok else '*** FAIL ***'
    print(f'{i:>2}  {c["expected"]:10s}  {got:10s}  {r["fraud_probability"]:.6f}  {flag:5s}  {c["note"]}')

print('-' * 80)
print(f'{passed}/{len(CASES)} cases passed')
if passed == len(CASES):
    print('ALL PASS - model behaves as expected, safe to deploy.')
else:
    print('Some FAIL - inspect those rows before deploying.')